# Loan Data Analysis

## Описание

В данной работе будет произведены анализ, обработка и обучение модели машинного обучения для реализации оценки дефолта заёмщика. Данные для этой работы взяты по [ссылке](https://ods.ai/tracks/dl_in_finance/competitions/dl-fintech-bki/data).


## Import libraries

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from collections import Counter
import sys

In [2]:
# Version of Python used
print('Python==' + str(sys.version))

Python==3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [3]:
# Version numbers of libraries used
print('pandas==' + str(pd.__version__))
print('numpy==' + str(np.__version__))
print('seaborn==' + str(sns.__version__))
print('scikit-learn==' + str(sys.modules[StandardScaler.__module__[:StandardScaler.__module__.index(".")]].__version__))

pandas==2.3.2
numpy==2.3.2
seaborn==0.13.2
scikit-learn==1.8.0


In [4]:
sns.set_context('notebook', font_scale = 1) 

## Данные
Датасет соревнования устроен таким образом, что кредиты для тренировочной выборки взяты за период в М месяцев, а кредиты для тестовой выборки взяты за последующие K месяцев.

Каждая запись кредитной истории содержит самую разнообразную информацию о прошлом кредите клиента, например, сумму, отношение клиента к кредиту, дату открытия и закрытия, информацию о просрочках по платежам и др. Все публикуемые данные тщательно анонимизированы.

Целевая переменная – бинарная величина, принимающая значения 0 и 1, где 1 соответствует дефолту клиента по кредиту.

Остальные поля можно увидеть по [ссылке](https://ods.ai/tracks/dl_in_finance/competitions/dl-fintech-bki/data).

In [ ]:
test_data = pd.read_parquet("data/test_data")
train_data = pd.read_parquet("data/train_data")

In [ ]:
test_data.count()

id                         26162717
rn                         26162717
pre_since_opened           26162717
pre_since_confirmed        26162717
pre_pterm                  26162717
                             ...   
enc_loans_credit_status    26162717
enc_loans_credit_type      26162717
enc_loans_account_cur      26162717
pclose_flag                26162717
fclose_flag                26162717
Length: 61, dtype: int64

In [13]:
train_data.count()

id                         26162717
rn                         26162717
pre_since_opened           26162717
pre_since_confirmed        26162717
pre_pterm                  26162717
                             ...   
enc_loans_credit_status    26162717
enc_loans_credit_type      26162717
enc_loans_account_cur      26162717
pclose_flag                26162717
fclose_flag                26162717
Length: 61, dtype: int64

In [8]:
test_data.head()

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,...,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,3000000,1,11,5,17,14,12,11,3,2,...,3,3,3,4,1,3,4,1,0,0
1,3000000,2,19,16,15,9,12,11,16,3,...,3,3,3,4,1,2,4,1,0,0
2,3000001,1,16,17,8,5,4,9,5,2,...,3,3,3,4,1,3,4,1,0,0
3,3000001,2,16,7,9,0,4,9,1,2,...,3,3,3,4,1,3,4,1,0,0
4,3000001,3,10,0,14,7,11,12,2,2,...,1,1,0,2,1,3,4,1,0,0


In [10]:
train_data.head()

,id,rn,pre_since_opened,pre_since_confirmed,pre_pterm,pre_fterm,pre_till_pclose,pre_till_fclose,pre_loans_credit_limit,pre_loans_next_pay_summ,...,enc_paym_21,enc_paym_22,enc_paym_23,enc_paym_24,enc_loans_account_holder_type,enc_loans_credit_status,enc_loans_credit_type,enc_loans_account_cur,pclose_flag,fclose_flag
0,0,1,18,9,2,3,16,10,11,3,...,3,3,3,4,1,3,4,1,0,0
1,0,2,18,9,14,14,12,12,0,3,...,0,0,0,4,1,3,4,1,0,0
2,0,3,18,9,4,8,1,11,11,0,...,0,0,0,4,1,2,3,1,1,1
3,0,4,4,1,9,12,16,7,12,2,...,3,3,3,4,1,3,1,1,0,0
4,0,5,5,12,15,2,11,12,10,2,...,3,3,3,4,1,3,4,1,0,0


In [11]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4724601 entries, 0 to 4724600
Data columns (total 61 columns):
 #   Column                         Dtype
---  ------                         -----
 0   id                             int64
 1   rn                             int64
 2   pre_since_opened               int64
 3   pre_since_confirmed            int64
 4   pre_pterm                      int64
 5   pre_fterm                      int64
 6   pre_till_pclose                int64
 7   pre_till_fclose                int64
 8   pre_loans_credit_limit         int64
 9   pre_loans_next_pay_summ        int64
 10  pre_loans_outstanding          int64
 11  pre_loans_total_overdue        int64
 12  pre_loans_max_overdue_sum      int64
 13  pre_loans_credit_cost_rate     int64
 14  pre_loans5                     int64
 15  pre_loans530                   int64
 16  pre_loans3060                  int64
 17  pre_loans6090                  int64
 18  pre_loans90                    int64
 19  

In [12]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26162717 entries, 0 to 26162716
Data columns (total 61 columns):
 #   Column                         Dtype
---  ------                         -----
 0   id                             int64
 1   rn                             int64
 2   pre_since_opened               int64
 3   pre_since_confirmed            int64
 4   pre_pterm                      int64
 5   pre_fterm                      int64
 6   pre_till_pclose                int64
 7   pre_till_fclose                int64
 8   pre_loans_credit_limit         int64
 9   pre_loans_next_pay_summ        int64
 10  pre_loans_outstanding          int64
 11  pre_loans_total_overdue        int64
 12  pre_loans_max_overdue_sum      int64
 13  pre_loans_credit_cost_rate     int64
 14  pre_loans5                     int64
 15  pre_loans530                   int64
 16  pre_loans3060                  int64
 17  pre_loans6090                  int64
 18  pre_loans90                    int64
 19

In [15]:
test_target_data = pd.read_csv("data/test_target.csv")
train_target_data = pd.read_csv("data/train_target.csv")

In [18]:
test_target_data.info()
print()
train_target_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   id      500000 non-null  int64
dtypes: int64(1)
memory usage: 3.8 MB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 2 columns):
 #   Column  Dtype
---  ------  -----
 0   id      int64
 1   flag    int64
dtypes: int64(2)
memory usage: 45.8 MB


In [21]:
test_target_data.head()

,id
0,3000000
1,3000001
2,3000002
3,3000003
4,3000004


In [22]:
train_target_data.head()

,id,flag
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


## Анализ важности полей для прогноза кредитного дефолта

### Группа 1: КРИТИЧЕСКИ ВАЖНЫЕ (⭐⭐⭐⭐⭐)
**Прямые индикаторы текущих и прошлых проблем.**

| Поле | Причина важности |
|------|------------------|
| `pre_loans_total_overdue` | Текущая сумма просрочки. Если > 0 — клиент уже в дефолте. |
| `pre_loans90` | Количество просрочек >90 дней. Сильнейший сигнал о прошлых серьёзных проблемах. |
| `pre_loans3060` | Просрочки 30-60 дней. Показатель хронических проблем с платежами. |
| `pre_maxover2limit` | Максимальная историческая просрочка относительно лимита. Показывает "дно" финансовой ямы клиента. |

### Группа 2: ОЧЕНЬ ВАЖНЫЕ (⭐⭐⭐⭐)
**Показатели финансовой нагрузки и поведения.**

| Поле | Причина важности |
|------|------------------|
| `pre_util` | Уровень использования кредитного лимита. Выше 0.8 = высокая нагрузка, высокий риск. |
| `pre_over2limit` | Текущая просрочка относительно лимита. Прямой индикатор остроты текущих проблем. |
| `enc_paym_0` ... `enc_paym_24` | История платежей за 25 месяцев. **Тренд** (ухудшение/улучшение) важнее единичных значений. |
| `pre_loans530` | Просрочки 5-30 дней. Индикатор начинающихся или частых мелких проблем. |

### Группа 3: ВАЖНЫЕ (⭐⭐⭐)
**Контекстные и описательные признаки.**

| Поле | Причина важности |
|------|------------------|
| `pre_loans_outstanding` | Общая сумма долга. В сочетании с доходом даёт Debt-to-Income ratio. |
| `pre_loans_credit_limit` | Общий лимит. Важен для расчёта `pre_util` и других соотношений. |
| `pre_loans_credit_cost_rate` | Полная стоимость кредита. Более дорогие кредиты часто несут больший риск. |
| `enc_loans_credit_type` | Тип кредита. Риск дефолта различается для ипотеки, автокредита и т.д. |
| `pre_since_opened` | "Возраст" кредита. Новые кредиты часто рискованнее. |

### Группа 4: ВСПОМОГАТЕЛЬНЫЕ/ФЛАГИ (⭐⭐)
**Бинарные индикаторы, полезные как фильтры.**

| Поле | Причина важности |
|------|------------------|
| `is_zero_loans90` | Флаг отсутствия серьёзных просрочек. `1` — хороший сигнал. |
| `is_zero_loans3060` | Флаг отсутствия средних просрочек. |
| `is_zero_util` | Флаг отсутствия долга (нулевая нагрузка). `1` — хороший сигнал. |
| `pclose_flag`, `fclose_flag` | Отсутствие плановой/фактической даты закрытия. Может указывать на открытые/бессрочные кредиты. |

### Группа 5: МАЛОВАЖНЫЕ / ТЕХНИЧЕСКИЕ (⭐)
**Слабая предсказательная сила или дублирование информации.**

| Поле | Причина низкой важности |
|------|--------------------------|
| `pre_loans5` | Просрочки до 5 дней часто не считаются значимыми или техническими. |
| `pre_loans6090` | Частично перекрывается признаками `pre_loans3060` и `pre_loans90`. |
| `pre_since_confirmed` | Скорее технический временной параметр. |
| `pre_pterm`, `pre_fterm` | Части информация содержится в `pre_till_pclose`/`pre_till_fclose`. |
| `enc_loans_account_cur` | Валюта счёта. В стабильной экономике слабый предиктор. |
| `rn` | Порядковый номер кредита. Служебное поле, не является фичей. |

### Группа 6: НЕ ИСПОЛЬЗОВАТЬ КАК ПРИЗНАК
**Идентификаторы, а не предикторы.**

| Поле | Причина |
|------|---------|
| `id` | Уникальный идентификатор клиента. Несёт в себе информацию, но его использование приведёт к переобучению. |

### КЛЮЧЕВОЙ ВЫВОД
**Сигналы дефолта идут в порядке убывания важности:**
1.  **Наличие прошлых серьёзных просрочек** (`pre_loans90`) — самый сильный сигнал.
2.  **Высокая текущая финансовая нагрузка** (`pre_util`, `pre_over2limit`).
3.  **Отрицательная динамика платежного поведения** (тренд по `enc_paym_*`).
4.  **Общий контекст** (тип и стоимость кредита, количество обязательств).

## Проверка данных